# 教師あり機械学習プロジェクト課題：動作確認ノートブック

このノートブックは、提出予定のPythonファイルが採点環境で正しく動作するかを確認するためのものである。

## 使用するファイル

次の3ファイルを用意すること。

- `train.csv`
- `test_sample.csv`
- 提出予定のPythonファイル（例：`24RB1001.py`）

## 注意

- `test_sample.csv` は動作確認用であり、その予測性能は評価しない。
- 本番採点では、教員側で非公開のテストデータを使用する。
- このノートブック自体は提出しない。
- 提出するのは、学生証番号をファイル名にしたPythonファイルとレポートPDFである。


## 1. ファイルのアップロード

以下のセルを実行し、次の3ファイルを選択する。

- `train.csv`
- `test_sample.csv`
- 提出予定のPythonファイル


In [ ]:
from google.colab import files

uploaded = files.upload()

print("アップロードされたファイル:")
for filename in uploaded:
    print(" -", filename)


## 2. 提出予定ファイル名の指定

`STUDENT_FILE` を、自分の提出予定ファイル名に変更すること。

例：

```python
STUDENT_FILE = "24RB1001.py"
```


In [ ]:
STUDENT_FILE = "24RB1001.py"  # 自分の学生証番号に変更する

TRAIN_FILE = "train.csv"
TEST_SAMPLE_FILE = "test_sample.csv"


## 3. 必要ファイルの確認

指定したファイルがすべて存在するかを確認する。


In [ ]:
from pathlib import Path

required_files = [
    Path(TRAIN_FILE),
    Path(TEST_SAMPLE_FILE),
    Path(STUDENT_FILE),
]

missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "次のファイルが見つかりません: " + ", ".join(missing_files)
    )

print("必要なファイルを確認しました。")


## 4. 提出予定のPythonファイルを読み込む

提出ファイルに `train_and_predict(train_path, test_path)` が定義されているかを確認する。


In [ ]:
import importlib.util
import sys

module_name = "student_solution"

# 同じ名前のモジュールが既に読み込まれている場合は削除する
if module_name in sys.modules:
    del sys.modules[module_name]

spec = importlib.util.spec_from_file_location(module_name, STUDENT_FILE)

if spec is None or spec.loader is None:
    raise ImportError(f"{STUDENT_FILE} を読み込めません。")

student_solution = importlib.util.module_from_spec(spec)
spec.loader.exec_module(student_solution)

if not hasattr(student_solution, "train_and_predict"):
    raise AttributeError(
        "提出ファイルに train_and_predict 関数が定義されていません。"
    )

if not callable(student_solution.train_and_predict):
    raise TypeError("train_and_predict は関数として定義してください。")

print("train_and_predict 関数を確認しました。")


## 5. 関数を実行する

`train.csv` と `test_sample.csv` のパスを与えて、予測結果を取得する。


In [ ]:
predictions = student_solution.train_and_predict(
    TRAIN_FILE,
    TEST_SAMPLE_FILE,
)

print("train_and_predict の実行が完了しました。")


## 6. 戻り値の形式を確認する

次の条件を確認する。

- 戻り値が一次元配列に変換できる。
- 予測数が `test_sample.csv` の行数と一致する。
- 予測値が0または1である。
- 欠損値を含まない。


In [ ]:
import numpy as np
import pandas as pd

predictions = np.asarray(predictions)

assert predictions.ndim == 1, (
    "戻り値は一次元配列にしてください。"
)

test_sample = pd.read_csv(TEST_SAMPLE_FILE)

assert len(predictions) == len(test_sample), (
    f"予測数が一致しません。"
    f"予測数={len(predictions)}, "
    f"test_sample.csvの行数={len(test_sample)}"
)

assert not pd.isna(predictions).any(), (
    "予測結果に欠損値が含まれています。"
)

unique_values = set(np.unique(predictions).tolist())

assert unique_values.issubset({0, 1}), (
    f"予測値は0または1にしてください。"
    f"現在の予測値={sorted(unique_values)}"
)

print("出力形式の確認に成功しました。")
print("予測数:", len(predictions))
print("予測値の種類:", sorted(unique_values))
print("先頭10件:", predictions[:10])


## 7. 実行時間を確認する

採点時には実行時間に上限を設ける。以下のセルで、おおよその実行時間を確認できる。

なお、このセルではモデルの学習を再度行うため、前のセルと同じ処理がもう一度実行される。


In [ ]:
import time

start_time = time.perf_counter()

_ = student_solution.train_and_predict(
    TRAIN_FILE,
    TEST_SAMPLE_FILE,
)

elapsed_time = time.perf_counter() - start_time

print(f"実行時間: {elapsed_time:.2f} 秒")


## 8. 確認完了

すべてのセルがエラーなく実行できれば、提出コードの基本的な形式は正しい。

提出前に、Pythonファイル名が次の形式になっていることを確認すること。

```text
学生証番号.py
```

例：

```text
24RB1001.py
```

このノートブックは提出しない。
